<a href="https://colab.research.google.com/github/toryor31oct/group15-Fun-Rai-Kwam-plod-Phai/blob/mail/SQL_Part_Mail_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

เมล


In [ ]:
import pandas as pd
import sqlite3

# 1. อ่านไฟล์ CSV (เปลี่ยนชื่อไฟล์เป็นชื่อไฟล์ของคุณ เช่น 'claims_system.csv')
df = pd.read_csv("claims_system_300.csv")

# 2. สร้างการเชื่อมต่อฐานข้อมูลในหน่วยความจำ (หรือตั้งชื่อไฟล์ .db)
conn = sqlite3.connect("my_insurance_db.db")

# 3. นำข้อมูลจาก CSV เขียนลงในตารางชื่อ claims_system
df.to_sql("claims_system", conn, if_exists="replace", index=False)

# 4. รันคำสั่ง SQL ตามต้องการ
q1_sql = """
SELECT
    insurance_type,
    SUM(net_payout) AS total_net_payout,
    AVG(claim_amount) AS avg_claim_amount
FROM claims_system
GROUP BY insurance_type
ORDER BY total_net_payout DESC;
"""

# 5. ดึงผลลัพธ์และแสดงผล
q1_result = pd.read_sql_query(q1_sql, conn)
display(q1_result)

,insurance_type,total_net_payout,avg_claim_amount
0,ประกันชีวิต,1.378575e+08,1.071699e+06
1,ประกันโรคร้ายแรง,8.197200e+07,6.648046e+05
2,ประกันสุขภาพ,2.621658e+07,4.698812e+05
3,ประกันอุบัติเหตุ,1.988678e+07,3.253208e+05


In [ ]:
# คำถามที่ 2: ภาพรวมสถานะการอนุมัติ (Approved vs Rejected) มีจำนวนกี่เคส และจ่ายเงินสุทธิรวมเท่าใด

q2_sql = """
SELECT
    approval_status,
    COUNT(claim_id) AS total_request,
    SUM(net_payout) AS total_payout
FROM claims_system
GROUP BY approval_status
ORDER BY total_request DESC;
"""

q2_result = pd.read_sql_query(q2_sql, conn)

display(q2_result)


,approval_status,total_request,total_payout
0,Approved,276,2.659329e+08
1,Rejected,24,0.000000e+00


In [ ]:
# คำถามที่ 3
# Part 1: จำนวนเคสที่ผ่านอนุมัติและถูกปฏิเสธ พร้อมยอดจ่ายรวม

q3_part1_sql = """
SELECT
    insurance_type,
    payment_status,
    COUNT(claim_id) AS total_cases,
    SUM(net_payout) AS total_payout
FROM claims_system
GROUP BY
    insurance_type,
    payment_status
ORDER BY
    insurance_type DESC,
    payment_status DESC;
"""

q3_part1_result = pd.read_sql_query(q3_part1_sql, conn)

display(q3_part1_result)


# Part 2: เหตุผลที่ไม่ผ่าน และยอดเงินที่ยื่นเกินวงเงิน

q3_part2_sql = """
SELECT
    insurance_type,
    claim_detail,
    COUNT(claim_id) AS rejected_cases,
    AVG(claim_amount) AS avg_claim_amount,
    AVG(coverage_limit) AS avg_coverage_limit,
    AVG(claim_amount - coverage_limit) AS avg_axcess_amount
FROM claims_system
WHERE payment_status = 'Cancelled'
GROUP BY
    insurance_type,
    claim_detail
ORDER BY
    rejected_cases DESC;
"""

q3_part2_result = pd.read_sql_query(q3_part2_sql, conn)

display(q3_part2_result)

,insurance_type,payment_status,total_cases,total_payout
0,ประกันโรคร้ายแรง,Paid,70,8.197200e+07
1,ประกันอุบัติเหตุ,Paid,68,1.988678e+07
2,ประกันอุบัติเหตุ,Cancelled,7,0.000000e+00
3,ประกันสุขภาพ,Paid,63,2.621658e+07
4,ประกันสุขภาพ,Cancelled,17,0.000000e+00
5,ประกันชีวิต,Paid,75,1.378575e+08


,insurance_type,claim_detail,rejected_cases,avg_claim_amount,avg_coverage_limit,avg_axcess_amount
0,ประกันสุขภาพ,ผ่าตัดไส้ติ่งอักเสบฉุกเฉิน,6,689911.156667,675000.000000,14911.156667
1,ประกันสุขภาพ,เข้ารับการรักษาผู้ป่วยใน (IPD) โรคไข้หวัดใหญ่,6,712611.008333,650000.000000,62611.008333
2,ประกันสุขภาพ,เข้ารับการรักษาผู้ป่วยนอก (OPD) ลำไส้อักเสบ,5,538794.148000,500000.000000,38794.148000
3,ประกันอุบัติเหตุ,เข้ารักษาฉุกเฉินจากลื่นล้มหกล้ม,4,537886.955000,512500.000000,25386.955000
4,ประกันอุบัติเหตุ,กระดูกข้อเท้าแตกหักจากการเล่นกีฬา,3,719952.443333,683333.333333,36619.110000


In [ ]:
# คำถามที่ 4: ผู้รับประโยชน์กลุ่มไหน (บิดา, มารดา, บุตร ฯลฯ) มีจำนวนเคสมากสุด พร้อมยอดจ่ายสูงสุดและค่าเฉลี่ย?
q4_sql = """
SELECT
    beneficiary_relationship,
    COUNT(claim_id) AS total_cases,
    MAX(net_payout) AS max_net_payout,
    AVG(net_payout) AS avg_net_payout
FROM claims_system
GROUP BY beneficiary_relationship
ORDER BY total_cases DESC;
"""

q4_result = pd.read_sql_query(q4_sql, conn)

display(q4_result)

,beneficiary_relationship,total_cases,max_net_payout,avg_net_payout
0,บุตร,60,2722500.0,866835.031333
1,สามี,51,2871000.0,719399.799412
2,ภรรยา,51,2970000.0,977530.774902
3,พี่น้อง,51,2920500.0,967573.841961
4,บิดา,45,2920500.0,948092.903111
5,มารดา,42,2920500.0,842115.515714
